# <h1 style='color: #1B3A6B;'>1. NOMBRE DEL PROYECTO</h1> 

## <span style='color: #2E86AB'>PREDICCIÓN DE ABANDONO DE CLIENTES — BETA BANK (CHURN)</span>

---
# <span style='color: #1B3A6B;'>2. OBJETIVO </span>
* Desarrollar un modelo de clasificación binaria que prediga si un cliente de Beta Bank abandonará el banco próximamente (**Exited = 1**) o no (**Exited = 0**), con base en su comportamiento e historial con el banco.
* **Criterio mínimo de aprobación:** F1 Score ≥ 0.59 en el conjunto de prueba.
* **Métrica secundaria:** AUC-ROC — se debe medir y comparar con el valor F1.

---
# <span style='color: #1B3A6B;'>3. DESCRIPCIÓN </span>

---
## <span style='color: #2E86AB;'>3.1 Descripción del Proyecto </span>

Los clientes de Beta Bank se están yendo, cada mes, poco a poco. Los banqueros descubrieron que es más barato salvar a los clientes existentes que atraer nuevos. Necesitamos predecir si un cliente dejará el banco pronto, usando datos sobre el comportamiento pasado de los clientes y la terminación de contratos. El umbral mínimo de F1 Score es 0.59.

## <span style='color: #2E86AB;'>3.2. Descripción del Dataset </span>

Cada observación en el dataset contiene información sobre el perfil y comportamiento de un cliente bancario. La información dada es la siguiente:

* **RowNumber** — índice de cadena de datos (no informativo),
* **CustomerId** — identificador de cliente único (no informativo),
* **Surname** — apellido del cliente (no informativo),
* **CreditScore** — valor de crédito del cliente,
* **Geography** — país de residencia del cliente,
* **Gender** — sexo del cliente,
* **Age** — edad del cliente,
* **Tenure** — años que lleva el cliente con el banco,
* **Balance** — saldo de la cuenta bancaria,
* **NumOfProducts** — número de productos bancarios utilizados,
* **HasCrCard** — ¿tiene tarjeta de crédito? (1 = sí, 0 = no),
* **IsActiveMember** — ¿es miembro activo? (1 = sí, 0 = no),
* **EstimatedSalary** — salario estimado del cliente.

**Variable objetivo:**
* **Exited** — el cliente se ha ido (1 = sí, 0 = no).

---
# <span style='color: #1B3A6B'>4. PREPARACIÓN DE DATOS </span>

## <span style='color: #2E86AB;'> 4.1. Importar Librerías </span>

In [ ]:
import pandas as pd                                               # Maneja y manipula datos en tablas
import numpy as np                                                # Operaciones numéricas y vectoriales
import matplotlib.pyplot as plt                                   # Librería para graficar
import seaborn as sns                                             # Librería para gráficos más estéticos
from sklearn.tree import DecisionTreeClassifier                   # Modelo de árbol de decisión
from sklearn.ensemble import RandomForestClassifier               # Modelo de bosque aleatorio
from sklearn.ensemble import GradientBoostingClassifier           # Modelo de gradient boosting
from sklearn.linear_model import LogisticRegression               # Modelo de regresión logística
from sklearn.preprocessing import StandardScaler                  # Escala las variables numéricas
from sklearn.model_selection import train_test_split              # Divide el dataset en train, validación, test
from sklearn.model_selection import GridSearchCV                  # Búsqueda automática y sistemática de hiperparámetros
from sklearn.metrics import f1_score                              # Métrica principal del proyecto
from sklearn.metrics import roc_auc_score                         # Métrica secundaria del proyecto
from sklearn.metrics import classification_report                 # Reporte completo de métricas por clase
from sklearn.metrics import confusion_matrix                      # Matriz de confusión para ver dónde falla el modelo
from sklearn.metrics import roc_curve                             # Curva ROC para graficar el AUC-ROC
from imblearn.over_sampling import SMOTE                          # Sobremuestreo sintético para balancear clases
from sklearn.utils import resample                                # Submuestreo para balancear clases

## <span style='color: #2E86AB;'>4.2. Cargar el Dataset</span>

In [ ]:
df = pd.read_csv('/datasets/Churn.csv')


## <span style='color: #2E86AB;'>4.3. Exploración Inicial de Datos </span>

In [ ]:
# Exploración EDA

print('--'*40)
print('                          INFORMACIÓN GENERAL DEL DATASET')
print('--'*40)
df.info()

print('--'*40)
print('                             DESCRIPCIÓN DEL DATASET')
print('--'*40)
print(df.describe())

print('--'*40)
print('                            NÚMERO DE FILAS Y COLUMNAS')
print('--'*40)
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')

print('--'*40)
print('                                 TIPOS DE DATOS')
print('--'*40)
print(df.dtypes)

print('--'*40)
print('                          PRIMERAS 5 FILAS DEL DATASET')
print('--'*40)
print(df.head())

## <span style='color: #2E86AB;'>4.4. Verificar Valores Ausentes y Duplicados </span>

In [ ]:
# Valores ausentes y duplicados
print('--'*40)
print('                  VALORES NULOS POR COLUMNA')
print('--'*40)
print(df.isnull().sum())

print('--'*40)
print('                    VALORES DUPLICADOS')
print('--'*40)
print(f'DUPLICADOS: {df.duplicated().sum()}')

print('--'*40)
print('             10 FILAS ALEATORIAS DEL DATASET')
print('--'*40)
print(df.sample(10))

## <span style='color:#2E86AB;'>4.5. Eliminación de Columnas Irrelevantes</span>

In [ ]:
# Eliminar columnas que no aportan valor predictivo al modelo

df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)     # RowNumber: índice; CustomerId y Surname: identificadores sin poder predictivo

print('--'*40)
print('              COLUMNAS RESTANTES TRAS ELIMINACIÓN')
print('--'*40)
print(df.columns.tolist())

## <span style='color:#2E86AB;'>4.6. Codificación de Variables Categóricas</span>

In [ ]:
# Convertir variables categóricas a numéricas mediante One-Hot Encoding
# Geography: Francia, Alemania, España → 3 columnas dummy
# Gender: Male / Female → 2 columnas dummy

df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True)    # drop_first evita multicolinealidad perfecta

print('--'*40)
print('              COLUMNAS TRAS CODIFICACIÓN OHE')
print('--'*40)
print(df.dtypes)

> **NOTA:** En la exploración EDA se puede observar lo siguiente:

- Se eliminaron 3 columnas irrelevantes: `RowNumber`, `CustomerId` y `Surname`,
- Se aplicó One-Hot Encoding a `Geography` y `Gender` para convertirlas en variables numéricas,
- Se verificará si existen valores nulos o duplicados antes de continuar.

Describe aquí tus hallazgos tras la exploración inicial.

---
# <span style='color: #1B3A6B;'> 5. DIVISIÓN DE DATOS </span>

## <span style='color: #2E86AB'>5.1. Separar Características (Features) y Objetivo (Target)</span>

In [ ]:
# Features y Target

features = df.drop(['Exited'], axis=1)       # Variables de entrada X: elimina la columna 'Exited'
target   = df['Exited']                      # Variable objetivo Y: columna que el modelo debe predecir

## <span style='color: #2E86AB;'>5.2. Dividir en Conjunto de Entrenamiento, Validación y Prueba</span>

In [ ]:
# Dividir los datos: 60% entrenamiento, 20% validación, 20% prueba

# 1. Dividir dataset en: Entrenamiento temporal (80%) y prueba (20%)

train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=12345
)

# 2. Dividir entrenamiento temporal en: Entrenamiento final (60%) y validación (20%)

train_features, valid_features, train_target, valid_target = train_test_split(
    train_features,
    train_target,
    test_size=0.25,
    random_state=12345
)

> **IMPORTANTE:**

Los datos son 'DIVIDIDOS' EN: 60% entrenamiento, 20% validación, 20% prueba.

## <span style='color: #2E86AB;'>5.3. Escalado de Características Numéricas</span>

In [ ]:
# Escalar features numéricas — IMPORTANTE: fit() solo sobre train para evitar data leakage

scaler = StandardScaler()

train_features = scaler.fit_transform(train_features)    # Ajusta el scaler con train y transforma
valid_features = scaler.transform(valid_features)        # Solo transforma — NO ajusta
test_features  = scaler.transform(test_features)         # Solo transforma — NO ajusta

---
# <span style='color:#1B3A6B;'>6. ANÁLISIS DEL EQUILIBRIO DE CLASES </span>

## <span style='color:#2E86AB;'>6.1. Distribución de la Variable Objetivo (Exited)</span>

In [ ]:
# Visualizar la distribución de clases en el conjunto de entrenamiento

conteo = pd.Series(train_target).value_counts()                           # Cuenta los valores de cada clase
ratio  = conteo[0] / conteo[1]                                            # Calcula el ratio de desequilibrio

print('='*50)
print('DISTRIBUCIÓN DE CLASES — CONJUNTO DE ENTRENAMIENTO')
print('='*50)
print(f'Clase 0 (No abandona): {conteo[0]} ({conteo[0]/len(train_target):.2%})')
print(f'Clase 1 (Abandona):    {conteo[1]} ({conteo[1]/len(train_target):.2%})')
print(f'Ratio de desequilibrio: {ratio:.2f}:1')
print('='*50)

# Graficar la distribución
conteo.plot(kind='bar', color=['#2E86AB', '#E84855'], edgecolor='black')
plt.title('Distribución de clases — Exited')
plt.xlabel('Clase (0 = No abandona, 1 = Abandona)')
plt.ylabel('Cantidad de clientes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## <span style='color:#2E86AB;'>6.2. Modelo Base sin Corrección de Desequilibrio</span>

In [ ]:
# Entrenar modelo base sin ningún ajuste de desequilibrio
# Objetivo: observar el impacto del desequilibrio sobre el F1 Score

modelo_base = RandomForestClassifier(random_state=12345, n_estimators=50)
modelo_base.fit(train_features, train_target)

pred_base = modelo_base.predict(valid_features)                           # Predicciones en validación
proba_base = modelo_base.predict_proba(valid_features)[:, 1]              # Probabilidades para AUC-ROC

f1_base  = f1_score(valid_target, pred_base)                              # F1 Score del modelo base
auc_base = roc_auc_score(valid_target, proba_base)                        # AUC-ROC del modelo base

print('='*50)
print('MODELO BASE — SIN CORRECCIÓN DE DESEQUILIBRIO')
print('='*50)
print(f'F1 Score  (validación): {f1_base:.4f} ({f1_base:.2%})')
print(f'AUC-ROC   (validación): {auc_base:.4f} ({auc_base:.2%})')
print(f'¿Cumple F1 ≥ 0.59?    {"SÍ" if f1_base >= 0.59 else "NO"}')
print('='*50)

> **Observaciones — Modelo Base**
>
> Describe aquí el impacto del desequilibrio de clases sobre el F1 Score.
> ¿El modelo base cumple el umbral de 0.59? ¿Por qué el desequilibrio perjudica al F1?

---
# <span style='color:#1B3A6B;'>7. MEJORA DEL MODELO — CORRECCIÓN DEL DESEQUILIBRIO </span>

## <span style='color: #2E86AB;'>7.1. Enfoque 1 — Ajuste de Pesos de Clase (class_weight='balanced')</span>

Se utiliza `class_weight='balanced'` para penalizar más los errores sobre la clase minoritaria (Exited=1), sin modificar los datos originales.

In [ ]:
# Árbol de Decisión con class_weight='balanced'

best_f1_dt    = 0
best_depth_dt = 0

for depth in range(1, 21):
    model_dt = DecisionTreeClassifier(random_state=12345, max_depth=depth, class_weight='balanced')
    model_dt.fit(train_features, train_target)
    pred_dt = model_dt.predict(valid_features)
    f1_dt   = f1_score(valid_target, pred_dt)

    if f1_dt > best_f1_dt:
        best_f1_dt    = f1_dt
        best_depth_dt = depth

print('='*50)
print('ÁRBOL DE DECISIÓN — class_weight=balanced')
print('='*50)
print(f'Mejor profundidad: {best_depth_dt}')
print(f'F1 Score (validación): {best_f1_dt:.4f} ({best_f1_dt:.2%})')
print('='*50)

In [ ]:
# Bosque Aleatorio con class_weight='balanced'

best_f1_rf  = 0
best_est_rf = 0

for est in range(10, 101, 10):
    model_rf = RandomForestClassifier(random_state=12345, n_estimators=est, class_weight='balanced')
    model_rf.fit(train_features, train_target)
    pred_rf = model_rf.predict(valid_features)
    f1_rf   = f1_score(valid_target, pred_rf)

    if f1_rf > best_f1_rf:
        best_f1_rf  = f1_rf
        best_est_rf = est

print('='*50)
print('BOSQUE ALEATORIO — class_weight=balanced')
print('='*50)
print(f'Mejor número de árboles: {best_est_rf}')
print(f'F1 Score (validación): {best_f1_rf:.4f} ({best_f1_rf:.2%})')
print('='*50)

In [ ]:
# Regresión Logística con class_weight='balanced'

model_lr = LogisticRegression(random_state=12345, solver='liblinear', class_weight='balanced')
model_lr.fit(train_features, train_target)

pred_lr = model_lr.predict(valid_features)
f1_lr   = f1_score(valid_target, pred_lr)

print('='*50)
print('REGRESIÓN LOGÍSTICA — class_weight=balanced')
print('='*50)
print(f'F1 Score (validación): {f1_lr:.4f} ({f1_lr:.2%})')
print('='*50)

> **Observaciones — Enfoque 1 (class_weight='balanced')**
>
> Describe aquí los resultados de los 3 modelos con class_weight y compáralos con el modelo base.

## <span style='color: #2E86AB;'>7.2. Enfoque 2 — Sobremuestreo con SMOTE</span>

Se utiliza `SMOTE` (Synthetic Minority Oversampling Technique) para generar muestras sintéticas de la clase minoritaria en el conjunto de entrenamiento. **IMPORTANTE:** SMOTE se aplica SOLO al conjunto de entrenamiento, nunca a validación ni a test.

In [ ]:
# Aplicar SMOTE al conjunto de entrenamiento

smote = SMOTE(random_state=12345)
train_features_smote, train_target_smote = smote.fit_resample(train_features, train_target)

print('='*50)
print('DISTRIBUCIÓN TRAS APLICAR SMOTE')
print('='*50)
print(pd.Series(train_target_smote).value_counts())
print('='*50)

In [ ]:
# Bosque Aleatorio entrenado con datos SMOTE

best_f1_smote  = 0
best_est_smote = 0

for est in range(10, 101, 10):
    model_smote = RandomForestClassifier(random_state=12345, n_estimators=est)
    model_smote.fit(train_features_smote, train_target_smote)             # Entrena con datos balanceados por SMOTE
    pred_smote = model_smote.predict(valid_features)                      # Evalúa en validación ORIGINAL (sin SMOTE)
    f1_smote   = f1_score(valid_target, pred_smote)

    if f1_smote > best_f1_smote:
        best_f1_smote  = f1_smote
        best_est_smote = est

print('='*50)
print('BOSQUE ALEATORIO — SMOTE')
print('='*50)
print(f'Mejor número de árboles: {best_est_smote}')
print(f'F1 Score (validación): {best_f1_smote:.4f} ({best_f1_smote:.2%})')
print('='*50)

> **Observaciones — Enfoque 2 (SMOTE)**
>
> Describe aquí los resultados con SMOTE y compáralos con el Enfoque 1.

## <span style='color: #2E86AB;'>7.3. Enfoque 3 — Submuestreo de la Clase Mayoritaria (Undersampling)</span>

In [ ]:
# Aplicar Undersampling: reducir la clase mayoritaria al tamaño de la clase minoritaria

train_df = pd.DataFrame(train_features)
train_df['target'] = train_target.values

clase_0 = train_df[train_df['target'] == 0]            # Clase mayoritaria (No abandona)
clase_1 = train_df[train_df['target'] == 1]            # Clase minoritaria (Abandona)

clase_0_under = resample(clase_0,                      # Submuestrea la clase mayoritaria
                         replace=False,
                         n_samples=len(clase_1),
                         random_state=12345)

train_under = pd.concat([clase_0_under, clase_1])      # Une ambas clases balanceadas

train_features_under = train_under.drop('target', axis=1).values
train_target_under   = train_under['target'].values

print('='*50)
print('DISTRIBUCIÓN TRAS UNDERSAMPLING')
print('='*50)
print(pd.Series(train_target_under).value_counts())
print('='*50)

# Entrenar Bosque Aleatorio con datos balanceados por Undersampling
model_under = RandomForestClassifier(random_state=12345, n_estimators=50)
model_under.fit(train_features_under, train_target_under)

pred_under = model_under.predict(valid_features)
f1_under   = f1_score(valid_target, pred_under)

print(f'F1 Score (validación): {f1_under:.4f} ({f1_under:.2%})')

> **Observaciones — Enfoque 3 (Undersampling)**
>
> Describe aquí los resultados con Undersampling y compáralos con los enfoques anteriores.

---
# <span style='color:#1B3A6B;'>8. SELECCIÓN DEL MEJOR MODELO </span>

## <span style='color:#2E86AB;'>8.1. Comparación de Resultados en Validación</span>

In [ ]:
# Tabla comparativa de todos los modelos y enfoques — ordenada por F1 Score

f1_umbral = 0.59

modelos = {
    'Base — Sin corrección'              : best_f1_base   if 'best_f1_base'   in dir() else f1_base,
    'Árbol Decisión — class_weight'      : best_f1_dt,
    'Bosque Aleatorio — class_weight'    : best_f1_rf,
    'Regresión Logística — class_weight' : f1_lr,
    'Bosque Aleatorio — SMOTE'           : best_f1_smote,
    'Bosque Aleatorio — Undersampling'   : f1_under
}

print('='*60)
print('COMPARACIÓN DE MODELOS — F1 SCORE EN VALIDACIÓN')
print('='*60)

for nombre, f1 in sorted(modelos.items(), key=lambda x: x[1], reverse=True):
    cumple = '✓' if f1 >= f1_umbral else '✗'
    print(f'{cumple}  {nombre:<45}: {f1:.4f} ({f1:.2%})')

print('='*60)
modelo_ganador = max(modelos.items(), key=lambda x: x[1])
print(f'Mejor modelo: {modelo_ganador[0]} con F1 = {modelo_ganador[1]:.2%}')
print('='*60)

## <span style='color:#2E86AB;'>8.2. Modelo Seleccionado y Justificación</span>

El modelo seleccionado es **[ESCRIBE AQUÍ EL MODELO GANADOR]** con **[X.XX%]** de F1 Score en validación:

**1. Cumplimiento del objetivo:**

* El F1 Score de validación de [X.XX%] supera el umbral mínimo requerido de 59%, cumpliendo con el requisito del proyecto.

**2. Superioridad comparativa:**
```
              MODELO                     F1 SCORE
       - [Modelo 1] — [Enfoque 1]  : XX.XX%
       - [Modelo 2] — [Enfoque 2]  : XX.XX%
       - [Modelo 3] — [Enfoque 3]  : XX.XX%
```
**3. Ventajas técnicas del modelo:**

Describe aquí las ventajas técnicas del modelo seleccionado.

## <span style='color:#2E86AB;'>8.3. Búsqueda Sistemática de Hiperparámetros (GridSearchCV)</span>

Se utiliza `GridSearchCV` para explorar de forma exhaustiva y automática múltiples combinaciones de hiperparámetros, usando `scoring='f1'` como criterio de optimización.

In [ ]:
sco# Búsqueda de hiperparámetros del mejor modelo con GridSearchCV

param_grid = {
    'n_estimators' : [50, 100, 150, 200],           # Número de árboles a evaluar
    'max_depth'    : [5, 10, 15, 20, None]           # Profundidades máximas a evaluar
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=12345, class_weight='balanced'),   # Modelo base a optimizar
    param_grid,                                      # Combinaciones de parámetros a probar
    cv=5,                                            # Divide los datos en 5 partes para validación cruzada
    scoring='f1',                                    # Métrica principal del proyecto
    n_jobs=-1                                        # Usa todos los núcleos del procesador para acelerar
)

grid_search.fit(train_features, train_target)        # Entrena todas las combinaciones posibles

print('='*50)
print('RESULTADO GRIDSEARCHCV')
print('='*50)
print(f'Mejores parámetros: {grid_search.best_params_}')
print(f'Mejor F1 Score:     {grid_search.best_score_:.4f} ({grid_search.best_score_:.2%})')
print('='*50)

> **Observaciones**
>
> `GridSearchCV` encontró que la mejor combinación es [n_estimators=X y max_depth=Y] con F1 de [XX.XX%].

---
# <span style='color:#1B3A6B;'>9. PRUEBA FINAL DEL MODELO</span>

## <span style='color:#2E86AB;'>9.1. Evaluación con el Conjunto de Prueba</span>

In [ ]:
# Entrenar el modelo final con los mejores hiperparámetros encontrados

final_model = RandomForestClassifier(
    random_state=12345,
    n_estimators=grid_search.best_params_['n_estimators'],
    max_depth=grid_search.best_params_['max_depth'],
    class_weight='balanced'
)

final_model.fit(train_features, train_target)                              # Entrena con el conjunto TRAIN

test_predictions = final_model.predict(test_features)                      # Predicciones en TEST
test_proba       = final_model.predict_proba(test_features)[:, 1]          # Probabilidades para AUC-ROC

## <span style='color:#2E86AB;'>9.2. Verificación del Umbral de F1 (F1 ≥ 0.59)</span>

In [ ]:
# Verificar si el modelo supera el umbral mínimo de F1 = 0.59

f1_test  = f1_score(test_target, test_predictions)
auc_test = roc_auc_score(test_target, test_proba)

print('='*50)
print('RESULTADO FINAL — CONJUNTO DE PRUEBA')
print('='*50)
print(f'F1 Score  (prueba): {f1_test:.4f} ({f1_test:.2%})')
print(f'AUC-ROC   (prueba): {auc_test:.4f} ({auc_test:.2%})')
print(f'¿Cumple F1 ≥ 0.59? {"SÍ" if f1_test >= 0.59 else "NO"}')
print('='*50)

## <span style='color:#2E86AB;'>9.3. Distribución Real vs Predicha</span>

In [ ]:
# Observar la distribución de predicciones del modelo final vs la distribución real

print('\nDistribución real (test):')
print(test_target.value_counts())

print('\nDistribución predicha (test):')
print(pd.Series(test_predictions).value_counts())

---
> **Observaciones**

Describe aquí las diferencias entre la distribución real y la predicha del conjunto de prueba.

## <span style='color:#2E86AB;'>9.4. Métricas Adicionales y Reporte de Clasificación</span>

El `F1 Score` equilibra Precision y Recall, siendo la métrica principal. El `AUC-ROC` mide la capacidad del modelo para distinguir entre clases, siendo útil para comparar.

In [ ]:
print('='*55)
print('MÉTRICAS ADICIONALES DEL MODELO FINAL')
print('='*55)

print(classification_report(
    test_target,
    test_predictions,
    target_names=['No abandona (0)', 'Abandona (1)']
))

print(f'F1 Score : {f1_test:.4f} ({f1_test:.2%})')
print(f'AUC-ROC  : {auc_test:.4f} ({auc_test:.2%})')
print('='*55)

> **Observaciones**

**Precision:**

- No abandona (0): [X.XX] → de cada 100 que predijo como "No abandona", [X] realmente no abandonaron
- Abandona (1):    [X.XX] → de cada 100 que predijo como "Abandona", [X] realmente abandonaron

**Recall:**
- No abandona (0): [X.XX] → de [N] clientes reales que no abandonan, detectó el [X]%
- Abandona (1):    [X.XX] → de [N] clientes reales que abandonan, detectó el [X]%

**F1-Score:**

- No abandona (0): [X.XX] → rendimiento en la clase mayoritaria
- Abandona (1):    [X.XX] → rendimiento en la clase minoritaria (la más importante)

## <span style='color:#2E86AB;'>9.5. Curva ROC y Comparativa F1 vs AUC-ROC</span>

In [ ]:
# Graficar la Curva ROC del modelo final

fpr, tpr, _ = roc_curve(test_target, test_proba)          # Tasa de falsos positivos y verdaderos positivos

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#2E86AB', lw=2, label=f'Curva ROC (AUC = {auc_test:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Modelo aleatorio (AUC = 0.50)')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC — Modelo Final')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('='*55)
print('COMPARATIVA F1 SCORE vs AUC-ROC')
print('='*55)
print(f'F1 Score : {f1_test:.4f} ({f1_test:.2%})  ← métrica principal')
print(f'AUC-ROC  : {auc_test:.4f} ({auc_test:.2%})  ← métrica secundaria')
print('='*55)

## <span style='color:#2E86AB;'>9.6. Matriz de Confusión</span>

In [ ]:
# Matriz de confusión del modelo final

cm = confusion_matrix(test_target, test_predictions)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No abandona', 'Abandona'],
            yticklabels=['No abandona', 'Abandona'])
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.title('Matriz de Confusión — Modelo Final')
plt.tight_layout()
plt.show()

## <span style='color:#2E86AB;'>9.7. Importancia de Variables</span>

In [ ]:
# Importancia de variables del modelo final

importancias = pd.Series(
    final_model.feature_importances_,
    index=df.drop('Exited', axis=1).columns
).sort_values(ascending=False)

print('='*50)
print('IMPORTANCIA DE VARIABLES')
print('='*50)
for variable, valor in importancias.items():
    print(f'{variable:<30}: {valor:.4f} ({valor:.2%})')
print('='*50)

importancias.plot(kind='bar', color='#2E86AB', edgecolor='black')
plt.title('Importancia de Variables — Modelo Final')
plt.ylabel('Importancia')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
# <span style='color:#1B3A6B; '>10. CONCLUSIONES </span>

---
**1.** De los modelos entrenados, el modelo elegido es el `[MODELO GANADOR]` con un F1 Score de validación de [XX.XX%], superando al resto de los modelos evaluados.

**2.** El desequilibrio de clases impactó significativamente al modelo base, que obtuvo un F1 Score de [XX.XX%] sin corrección. Tras aplicar `[MEJOR ENFOQUE]`, el F1 Score mejoró a [XX.XX%].

**3.** Con la selección del mejor modelo, se realizó una búsqueda sistemática de hiperparámetros con `GridSearchCV` (scoring='f1'), encontrando la mejor combinación [n_estimators=X, max_depth=Y].

**4.** El modelo final obtuvo un F1 Score de `[XX.XX%]` en el conjunto de prueba, superando el umbral mínimo requerido de 59%.

**5.** Comparativa entre las métricas principales:
```
- F1 Score : XX.XX%  ← métrica principal (equilibrio Precision-Recall)
- AUC-ROC  : XX.XX%  ← métrica secundaria (capacidad discriminatoria general)
```

**6.** Las `variables más importantes` para predecir el abandono fueron:

* `[Variable 1]` → [XX.XX%]  ← **la más determinante**
* `[Variable 2]` → [XX.XX%]
* `[Variable 3]` → [XX.XX%]
* `[Variable 4]` → [XX.XX%]  ← la menos determinante

---
## **RECOMENDACIONES A BETA BANK**

---
**1. Implementar el modelo:** Con [XX.XX%] de F1 Score el modelo está listo para identificar clientes en riesgo de abandono y permitir acciones preventivas.

**2. Priorizar los Falsos Negativos:** Un cliente que abandona y no fue detectado (Falso Negativo) representa un costo mayor que una alerta innecesaria. Se recomienda ajustar el umbral de decisión por debajo de 50% para capturar más clientes en riesgo.

**3. Enfocarse en variables clave:** Las variables con mayor peso predictivo deben ser monitoreadas activamente para detectar señales tempranas de abandono.

**4. Reevaluar el modelo periódicamente:** El comportamiento de los clientes cambia con el tiempo, por lo que se recomienda reentrenar el modelo cada 6 meses con datos actualizados.

**5. Ampliar el dataset:** Con más datos históricos, especialmente de clientes que abandonaron, el modelo podría mejorar su capacidad de detección y reducir los Falsos Negativos.